# 03. 에이전트 행동 경계 설계

## 목표
K3의 명시된 '과도한 주도성' 위험을 줄이기 위한 간단한 정책 검사기를 구현합니다. 실제 운영에서는 권한 샌드박스와 사람의 승인을 함께 사용해야 합니다.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Decision(Enum):
    ALLOW = "allow"
    ASK = "ask_user"
    DENY = "deny"

@dataclass
class Action:
    name: str
    destructive: bool = False
    external_side_effect: bool = False
    target_is_explicit: bool = True

def evaluate(action: Action) -> Decision:
    # 파괴적이고 대상도 불명확한 행동은 확인만으로 부족하므로 거부합니다.
    if action.destructive and not action.target_is_explicit:
        return Decision.DENY
    # 외부 전송이나 명확한 파괴 작업은 실행 전에 사람에게 확인합니다.
    if action.destructive or action.external_side_effect:
        return Decision.ASK
    return Decision.ALLOW

In [ ]:
actions = [
    Action("테스트 실행"),
    Action("고객에게 이메일 전송", external_side_effect=True),
    Action("명시된 임시 파일 삭제", destructive=True),
    Action("알 수 없는 경로 재귀 삭제", destructive=True, target_is_explicit=False),
]
for action in actions:
    print(f"{action.name}: {evaluate(action).value}")

## 확장 과제
1. 비용 한도, 실행 시간, 허용 디렉터리와 네트워크 도메인 규칙을 추가하세요.
2. `ASK` 결과에 사용자에게 보여줄 정확한 대상과 예상 영향을 포함하세요.
3. 행동 결정과 결과를 append-only 감사 로그로 남기세요.
4. 모델이 세션 중 바뀔 때 사고 이력 호환성을 검사하는 상태 필드를 설계하세요.

핵심 원칙은 모델의 말이 아니라 실행 계층에서 권한을 강제하는 것입니다.